## 机器人听觉感知系统：面试讲解稿

这份 notebook 用于 10 到 15 分钟面试讲解，重点突出项目定位、系统链路、技术难点、设计取舍和扩展性。

### 第 1 分钟：项目定位



#### 如果先用一句话定义这个项目，我会这样说：



这是一个机器人听觉感知系统，我想解决的不只是“把语音识别成文字”，而是让机器人完成从声音输入、身份确认、语义理解，到动作执行和语音反馈的完整闭环。



#### 如果再展开一点，我会补一句：



很多语音项目停留在 ASR 这一层，也就是把音频转成文本。但真正接近机器人应用时，系统还必须回答几个问题：是谁在说话、说的内容是不是有效指令、执行之后怎么反馈、反馈时又怎么避免误触发自己。所以我做这个项目时，目标不是单点功能，而是一个更完整、可运行的听觉交互链路。



<span style="color:red;">这段的核心作用，是让面试官一开始就知道我做的是系统整合型项目，而不是单一算法练习。</span>

### 第 2 到 4 分钟：整体流程



#### 整个系统的主流程是：

`语音采集 -> 声纹校验 -> ASR -> NLU -> 控制执行 -> TTS反馈`



第一步是语音采集，也就是通过麦克风实时获取用户音频输入。


第二步是声纹校验。我没有直接把所有输入都送去识别，而是先做 `Speaker Verification`。这样系统能够先判断“这个声音是不是已录入的目标用户”，只有通过校验的音频才会进入后续控制流程。这样做的好处是提高安全性，也减少无关人声对系统的干扰。

第三步是 `ASR`，也就是自动语音识别。我目前是通过 `SpeechRecognition` 调用 Google Web Speech API，把音频转成中文文本。

第四步是 `NLU`，也就是自然语言理解。这里我没有把它做成开放式大模型理解，而是针对机器人控制场景，做了规则化解析。比如系统能识别“前进”“后退”“左转”，也能进一步抽取“1米”“30度”“慢慢”“快速”这类参数，还能把“前进1m然后左转30度再前进1m”切成多段动作。


第五步是控制执行。解析完成后，系统把命令转成结构化动作，在当前项目里会驱动 GUI 里的 3D 小车运动；如果后续接真实机器人，也可以把这里替换成 ROS、串口或者 Socket 接口。


最后一步是 `TTS` 反馈，也就是机器人执行之后再用语音告诉用户“收到，正在前进”之类的结果。这样整个系统就形成了从输入到输出的闭环。


<span style="color:red;">这里可以补一句很关键的话：

所以这个项目的重点不是某一个模型有多先进，而是把多个能力模块串成了一个稳定可运行的系统。</span>

### 第 5 到 8 分钟：两个技术难点


 1. 自干扰抑制

这是很值得重点讲的一个点，因为它最能体现工程思维。

机器人语音系统里一个很实际的问题是自干扰，也就是机器人刚播报完一句话，麦克风又把这句话采集回来了，系统误以为是用户的新指令。严重的时候，系统会进入“自己对自己说话”的循环。

这个问题看起来简单，但实际上如果只靠“播报后延迟 1 秒再监听”是不稳的。因为不同长度的播报、不同环境的回声、不同麦克风灵敏度，都会导致误触发情况不一样。

所以我做了多层保护，而不是只做一个固定延时。

第一层是状态级保护。只要系统处于 `TTS` 播报状态，就直接暂停监听，不再调用麦克风采集。

第二层是时间级保护。播报结束后，不是立刻恢复监听，而是进入一个 `cooldown` 窗口，让尾音和环境回声先过去。

第三层是内容级保护。我会记录最近播报过的文本，如果后续识别出的文本和最近播报内容高度相似，就认为这是回声而不是用户真实输入，然后直接忽略。


如果面试官继续问为什么这样设计，我会补一句：

因为误触发本质上不是单一维度的问题，它同时跟时序、内容、环境都有关系。所以我用的是“状态控制 + 时间窗口 + 文本相似度过滤”的组合方案，这样鲁棒性会比单一延时更好。



#### 2. 声纹校验

第二个我重点处理的难点是声纹校验，也就是 `Speaker Verification`。

在很多语音 demo 里，谁说话系统都响应，但如果这是个机器人控制系统，其实会有一个隐含问题，就是权限控制。比如周围的人随便说一句“前进”，机器人就动了，这在真实场景里其实是不合理的。

所以我引入了声纹模块。实现上，我用的是 `PaddleSpeech` 里的 `ECAPA-TDNN` 模型。第一次运行时，系统会引导用户录入 5 条固定语音样本，然后提取每条样本的 speaker embedding，建立本地声纹档案。

后续每次有新的语音输入，系统会先提取这段语音的 embedding，再和已录入样本做相似度比较。我这里不是只做一次简单余弦相似度，而是综合了样本中心相似度、最近邻样本相似度和分布距离，来判断这个说话人是不是同一个人。



我也会补一句更专业的话：

也就是说，这里不是单纯在做“语音识别”，而是在做说话人身份验证，这是两个不同任务。前者解决“说了什么”，后者解决“是谁在说”。

### 第 9 到 11 分钟：设计取舍


#### 1. 为什么车控不用大模型直接生成动作

我没有让大模型直接解析车控命令，主要是因为机器人控制属于高确定性场景，它对稳定性、可解释性和低延迟要求更高。

比如“前进1米然后左转30度”，这种命令本质上是结构化的。如果交给规则解析，其实可以非常稳定地抽取出动作类型、距离和角度，而且执行结果是可预期的。

但如果交给大模型，虽然它在语言理解上更灵活，但会带来几个问题：

- 延迟更高
- 成本更高
- 输出不一定严格可控
- 调试难度更大

所以对机器人控制来说，我更倾向于用规则解析或者受限语法解析，让控制链路保持 `deterministic`，也就是确定性。



#### 2. 为什么问答模式又接大模型

但问答模式和控制模式不一样。问答属于开放域交互，它的重点不是严格执行动作，而是理解自然语言、组织表达、保持对话流畅性。在这个场景里，大模型的优势就很明显了。

所以我的做法其实是分层的：

- 控制命令走规则链路
- 开放问答走大模型链路

这是一种典型的“确定性控制 + 生成式交互”分层设计。这样既保证了机器人动作的稳定性，也保留了系统在问答场景下的自然交互能力。



#### 3. 为什么做模块化设计

我在项目里比较强调模块化，因为这个系统本质上是多能力拼接：麦克风采集、ASR、Speaker Verification、NLU、控制、TTS、GUI，这些模块未来都可能替换。

所以我在结构上把它们拆开了：

- `listener.py` 负责采集和识别
- `voiceprint.py` 负责声纹逻辑
- `config.py` 负责配置和解析规则
- `controller.py` 负责控制闭环
- `tts.py` 负责语音反馈和监听保护
- `gui_app.py` 负责 3D 展示和交互

这样拆分的好处是，如果后续我要把云端 ASR 换成本地 ASR，或者把 GUI 替换成真实机器人控制界面，不需要重写整个系统，只需要替换局部模块。



我会用一句话收束：

我希望这个项目不是一次性 demo，而是具备一定演进能力的原型系统。

### 第 12 到 15 分钟：扩展性和落地性



如果接真实机器人，我会优先改控制层。现在 `controller` 里默认是驱动 GUI 小车运动，后续可以把动作接口替换成串口控制、ROS topic 发布，或者 Socket 指令下发，这样就能和真实移动机器人底盘对接。

如果要做离线化，我会优先替换 ASR 和 TTS。现在 ASR 依赖 Google Web Speech，TTS 在不同平台也部分依赖系统能力。后续可以切到本地语音识别框架和本地语音合成模块，这样系统对网络的依赖会更低，更适合部署在边缘设备上。

如果要继续提升智能性，还可以加入多传感器融合。比如结合摄像头视觉输入、IMU、里程计、激光雷达，让机器人不仅能听懂命令，还能根据环境状态决定执行策略。



最后我会落到一句比较稳的话：

所以从架构上讲，这个项目虽然是课程项目起步，但它已经具备了往真实机器人交互原型演进的基础。

## 面试时可以直接使用的一段总结

这个项目最核心的价值，在于它不是把 ASR、声纹识别或者问答孤立地做出来，而是把这些能力整合成了一个可以真实跑起来的机器人听觉闭环系统。  
我在实现过程中比较关注两件事。第一件事是系统稳定性，比如怎么避免 TTS 误触发、怎么做重复命令去重、怎么保证车控链路可预测。第二件事是系统可扩展性，所以我把 ASR、Speaker Verification、NLU、Control、TTS、GUI 都做了相对清晰的模块拆分。  
我觉得这个项目比较能体现我的一个特点，就是我不只是关注某个算法点，而是更关注如何把多个能力拼起来，做成一个真正可运行、可迭代的系统。

## 面试官继续追问时的展开回答

### 1. 为什么选择 `ECAPA-TDNN` 做声纹识别

因为它在说话人识别任务里是比较成熟、效果稳定的一类架构。对我这个项目来说，我更看重的是“开源可用、社区成熟、能较方便集成到 Python 项目里”，而不是自己从零训练一个声纹模型。`PaddleSpeech` 对这一块封装得比较完整，所以我选择它作为工程实现方案。

### 2. 自干扰抑制为什么要做多层保护，而不是只加一个延时

因为误触发不只是时间问题。如果只靠固定延时，短播报和长播报的效果会不一样，环境回声也不一样。有时候即使过了延时，识别内容仍然可能是刚刚播报过的话。所以我把它拆成状态控制、冷却窗口和文本相似度过滤三个层次，覆盖不同类型的误触发来源。

### 3. 连续动作命令是怎么切分和解析的

我的做法是先对文本做归一化，再根据“然后、再、接着”等连接词切分成多个 segment。每个 segment 再单独做动作类型识别，并提取距离、角度和速度修饰，最后生成结构化命令序列。这样系统就能把一句长指令拆成多个可执行动作。

### 4. 为什么控制场景更适合规则解析而不是纯 LLM

因为控制场景强调的是可预测性和安全性。规则解析虽然不如 LLM 灵活，但在动作集合固定的前提下，结果更稳定，也更容易做边界控制。大模型更适合开放式问答，而不是直接承担底层动作决策。

### 5. 如果接真实机器人，通信层准备怎么改

我会保留现有的高层控制接口，把底层动作执行从“本地模拟”替换成“真实通信”。比如把 `forward`、`turn_left` 这类结构化命令映射成 ROS 消息、串口协议或者 TCP 指令。也就是说，上层的感知和理解模块可以不动，主要改控制适配层。

### 6. 如果做成离线系统，ASR 和 TTS 分别怎么替换

ASR 可以替换为本地语音识别引擎，TTS 可以替换为离线语音合成模块。这样做的意义主要有两个：一是降低对外部网络的依赖，二是提升隐私性和部署独立性。缺点是本地模型通常对算力和部署环境要求更高，需要在效果、资源和实时性之间做平衡。

## 相关代码入口

- [main.py](/Users/xuzai/Desktop/大学/大三下/机器人传感/robot_auditory/main.py)
- [gui_app.py](/Users/xuzai/Desktop/大学/大三下/机器人传感/robot_auditory/gui_app.py)
- [listener.py](/Users/xuzai/Desktop/大学/大三下/机器人传感/robot_auditory/listener.py)
- [voiceprint.py](/Users/xuzai/Desktop/大学/大三下/机器人传感/robot_auditory/voiceprint.py)
- [config.py](/Users/xuzai/Desktop/大学/大三下/机器人传感/robot_auditory/config.py)
- [tts.py](/Users/xuzai/Desktop/大学/大三下/机器人传感/robot_auditory/tts.py)